In [1]:
# Variaveis de sessão e organização de credenciais de usuário senhas e sessões.

spark.conf.set(
    "spark.sql.catalog.aicat.s3.access-key-id",
    "9340b73b9fa4150db3d6d85238faf7912a8fac1f"
)

spark.conf.set(
    "spark.sql.catalog.aicat.s3.secret-access-key",
    aidputils.secrets.get(name="SECRET_KEY_OBS", key="SECRET_KEY")
)
 
spark.conf.set(
    "spark.sql.catalog.aicat.credential",
    aidputils.secrets.get(name="ICEBERG_REST_CATALOG", key="USER:PASSWORD")
)

spark.conf.set(
    "spark.sql.catalog.aicat.scope",
    "PRINCIPAL_ROLE:ALL"
)


# Nome lógico do catálogo no Spark
spark.conf.set(
    "spark.sql.catalog.aicat",
    "org.apache.iceberg.spark.SparkCatalog"
)

# Tipo de catálogo
spark.conf.set(
    "spark.sql.catalog.aicat.type",
    "rest"
)

# Endpoint REST do Oracle AI Data Catalog
spark.conf.set(
    "spark.sql.catalog.aicat.uri",
    "https://G84191C2534E979-POCLAKEHOUSEICEBERG.adb.sa-saopaulo-1.oraclecloudapps.com/catalog"
)

# Nome do warehouse do catálogo Iceberg
spark.conf.set(
    "spark.sql.catalog.aicat.warehouse",
    "aicat_iceberg_rest_catalog"
)

# Autenticação OAuth 2.0
spark.conf.set(
    "spark.sql.catalog.aicat.rest.auth.type",
    "oauth2"
)

spark.conf.set(
    "spark.sql.catalog.aicat.oauth2-server-uri",
    "https://G84191C2534E979-POCLAKEHOUSEICEBERG.adb.sa-saopaulo-1.oraclecloudapps.com/catalog/v1/auth/token"
)

print("Configuração do catálogo REST aplicada.")

Configuração do catálogo REST aplicada.


In [2]:
landing_file = (
    "oci://poc-lakehouse-oci-landing@grwajeg8655r/"
    "inep/censo-escolar/2024/original/"
    "microdados_ed_basica_2024.csv"
)

jvm = spark._jvm
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

path = jvm.org.apache.hadoop.fs.Path(landing_file)
fs = path.getFileSystem(hadoop_conf)

print("Arquivo encontrado:", fs.exists(path))

if fs.exists(path):
    print("Tamanho (bytes):", fs.getFileStatus(path).getLen())

Arquivo encontrado: True
Tamanho (bytes): 217925280


In [3]:
landing_file = (
    "oci://poc-lakehouse-oci-landing@grwajeg8655r/"
    "inep/censo-escolar/2024/original/"
    "microdados_ed_basica_2024.csv"
)

df_censo = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("encoding", "windows-1252")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .load(landing_file)
)

print("Quantidade de colunas:", len(df_censo.columns))

print("\nColunas:")
print(df_censo.columns)

print("\nAmostra:")
df_censo.show(5, truncate=False)

Quantidade de colunas: 426

Colunas:
["NU_ANO_CENSO", "NO_REGIAO", "CO_REGIAO", "NO_UF", "SG_UF", "CO_UF", "NO_MUNICIPIO", "CO_MUNICIPIO", "NO_REGIAO_GEOG_INTERM", "CO_REGIAO_GEOG_INTERM", "NO_REGIAO_GEOG_IMED", "CO_REGIAO_GEOG_IMED", "NO_MESORREGIAO", "CO_MESORREGIAO", "NO_MICRORREGIAO", "CO_MICRORREGIAO", "NO_DISTRITO", "CO_DISTRITO", "NO_ENTIDADE", "CO_ENTIDADE", "TP_DEPENDENCIA", "TP_CATEGORIA_ESCOLA_PRIVADA", "TP_LOCALIZACAO", "TP_LOCALIZACAO_DIFERENCIADA", "DS_ENDERECO", "NU_ENDERECO", "DS_COMPLEMENTO", "NO_BAIRRO", "CO_CEP", "NU_DDD", "NU_TELEFONE", "TP_SITUACAO_FUNCIONAMENTO", "CO_ORGAO_REGIONAL", "DT_ANO_LETIVO_INICIO", "DT_ANO_LETIVO_TERMINO", "IN_VINCULO_SECRETARIA_EDUCACAO", "IN_VINCULO_SEGURANCA_PUBLICA", "IN_VINCULO_SECRETARIA_SAUDE", "IN_VINCULO_OUTRO_ORGAO", "IN_PODER_PUBLICO_PARCERIA", "TP_PODER_PUBLICO_PARCERIA", "IN_FORMA_CONT_TERMO_COLABORA", "IN_FORMA_CONT_TERMO_FOMENTO", "IN_FORMA_CONT_ACORDO_COOP", "IN_FORMA_CONT_PRESTACAO_SERV", "IN_FORMA_CONT_COOP_TEC_FIN", "IN

+------------+---------+---------+--------+-----+-----+---------------------+------------+---------------------+---------------------+-------------------+-------------------+-----------------+--------------+---------------+---------------+---------------------+-----------+----------------------------+-----------+--------------+---------------------------+--------------+---------------------------+-------------------------+-----------+---------------+-----------+--------+------+-----------+-------------------------+-----------------+--------------------+---------------------+------------------------------+----------------------------+---------------------------+----------------------+-------------------------+-------------------------+----------------------------+---------------------------+-------------------------+----------------------------+--------------------------+---------------------------+----------------------------+------------------------------+----------------------------+

In [4]:
from pyspark.sql import functions as F

# Total de registros
total = df_censo.count()

# Escolas distintas
escolas_distintas = (
    df_censo
    .select("CO_ENTIDADE")
    .distinct()
    .count()
)

# Registros sem código de escola
sem_codigo = (
    df_censo
    .filter(
        F.col("CO_ENTIDADE").isNull() |
        (F.trim(F.col("CO_ENTIDADE")) == "")
    )
    .count()
)

print("Total de registros:", total)
print("Escolas distintas:", escolas_distintas)
print("Registros sem código de escola:", sem_codigo)

Total de registros: 215545
Escolas distintas: 215545
Registros sem código de escola: 0


In [41]:
# Gerar automaticamente a definição das 426 colunas
colunas_sql = ",\n".join(
    f"`{col}` STRING"
    for col in df_censo.columns
)

ddl = f"""
CREATE TABLE IF NOT EXISTS aicat.BRONZE.CENSO_ESCOLAR_2024 (
    {colunas_sql}
)
USING iceberg
"""

spark.sql(ddl)

print("Tabela CENSO_ESCOLAR_2024 criada!")

Tabela CENSO_ESCOLAR_2024 criada!


In [43]:
df_censo.createOrReplaceTempView("vw_censo_escolar_2024")

spark.sql("""
    INSERT OVERWRITE aicat.BRONZE.CENSO_ESCOLAR_2024
    SELECT *
    FROM vw_censo_escolar_2024
""")

print("Carga Bronze concluída!")

Carga Bronze concluída!


In [48]:
tabelas = spark.sql("SHOW TABLES IN aicat.BRONZE")
tabelas.show(truncate=False)

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|BRONZE   |CENSO_ESCOLAR_2024|false      |
|BRONZE   |DS_BENEFICIO      |false      |
+---------+------------------+-----------+



In [49]:
from pyspark.sql import functions as F

df_bronze = spark.table("aicat.BRONZE.CENSO_ESCOLAR_2024")

print("Registros:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))
print(
    "Escolas distintas:",
    df_bronze.select("CO_ENTIDADE").distinct().count()
)
print(
    "Registros sem código de escola:",
    df_bronze.filter(
        F.col("CO_ENTIDADE").isNull() |
        (F.trim(F.col("CO_ENTIDADE")) == "")
    ).count()
)

Registros: 215545
Colunas: 426


Escolas distintas: 215545


Registros sem código de escola: 0


In [50]:
spark.sql("""
    SELECT
        file_path,
        file_format,
        record_count,
        file_size_in_bytes
    FROM aicat.BRONZE.CENSO_ESCOLAR_2024.files
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|file_path                                                                                                                 |file_format|record_count|file_size_in_bytes|
+--------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|s3://poc-lakehouse-oci-bronze/BRONZE/CENSO_ESCOLAR_2024/data/00000-39-eb553530-ab4b-4701-868d-187deb63b12b-0-00001.parquet|PARQUET    |56185       |8137146           |
|s3://poc-lakehouse-oci-bronze/BRONZE/CENSO_ESCOLAR_2024/data/00001-40-eb553530-ab4b-4701-868d-187deb63b12b-0-00001.parquet|PARQUET    |56332       |8437919           |
|s3://poc-lakehouse-oci-bronze/BRONZE/CENSO_ESCOLAR_2024/data/00002-41-eb553530-ab4b-4701-868d-187deb63b12b-0-00001.parquet|PARQUET    |54127       |831355

In [51]:
spark.sql("""
    SELECT
        committed_at,
        snapshot_id,
        operation,
        summary
    FROM aicat.BRONZE.CENSO_ESCOLAR_2024.snapshots
""").show(truncate=False)

+-----------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                        

In [52]:
df_bronze = spark.table("aicat.BRONZE.CENSO_ESCOLAR_2024")

for i, coluna in enumerate(df_bronze.columns, start=1):
    print(f"{i:03d} - {coluna}")

001 - NU_ANO_CENSO
002 - NO_REGIAO
003 - CO_REGIAO
004 - NO_UF
005 - SG_UF
006 - CO_UF
007 - NO_MUNICIPIO
008 - CO_MUNICIPIO
009 - NO_REGIAO_GEOG_INTERM
010 - CO_REGIAO_GEOG_INTERM
011 - NO_REGIAO_GEOG_IMED
012 - CO_REGIAO_GEOG_IMED
013 - NO_MESORREGIAO
014 - CO_MESORREGIAO
015 - NO_MICRORREGIAO
016 - CO_MICRORREGIAO
017 - NO_DISTRITO
018 - CO_DISTRITO
019 - NO_ENTIDADE
020 - CO_ENTIDADE
021 - TP_DEPENDENCIA
022 - TP_CATEGORIA_ESCOLA_PRIVADA
023 - TP_LOCALIZACAO
024 - TP_LOCALIZACAO_DIFERENCIADA
025 - DS_ENDERECO
026 - NU_ENDERECO
027 - DS_COMPLEMENTO
028 - NO_BAIRRO
029 - CO_CEP
030 - NU_DDD
031 - NU_TELEFONE
032 - TP_SITUACAO_FUNCIONAMENTO
033 - CO_ORGAO_REGIONAL
034 - DT_ANO_LETIVO_INICIO
035 - DT_ANO_LETIVO_TERMINO
036 - IN_VINCULO_SECRETARIA_EDUCACAO
037 - IN_VINCULO_SEGURANCA_PUBLICA
038 - IN_VINCULO_SECRETARIA_SAUDE
039 - IN_VINCULO_OUTRO_ORGAO
040 - IN_PODER_PUBLICO_PARCERIA
041 - TP_PODER_PUBLICO_PARCERIA
042 - IN_FORMA_CONT_TERMO_COLABORA
043 - IN_FORMA_CONT_TERMO_FOMENTO
044